# 02 — Pré-processamento sem contaminação do teste

A seleção de colunas é explícita. Índice serializado e `risk_level` nunca entram nos preditores.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src/config.py').is_file():
    raise RuntimeError('Abra o notebook a partir da raiz ou da pasta notebooks.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
from src.config import ExperimentConfig, TARGET
config = ExperimentConfig(output=ROOT)


In [ ]:
from src.train import prepare_data
from src.preprocessing import make_preprocessor
frame, split = prepare_data(config)
X_train, y_train = split['X_train'], split['y_train']
print('Classes de treino:', y_train.value_counts().to_dict())
print('Dimensão de treino:', X_train.shape)
assert set(split['train_indices']).isdisjoint(split['test_indices'])

Mediana para números e moda para o código meteorológico. One-hot evita ordem artificial. O arredondamento determinístico dos códigos meteorológicos elimina ruído decimal e é compartilhado com a inferência. Random Forest não requer normalização; `scale=True` permite modelos futuros. Este ajuste demonstrativo usa só o treino e não é reutilizado no tuning: cada fold ajusta um Pipeline novo.

In [ ]:
preprocessor = make_preprocessor()
transformed = preprocessor.fit_transform(X_train)
display(pd.DataFrame(transformed, columns=preprocessor.get_feature_names_out(), index=X_train.index).head())
print('Medianas aprendidas:', preprocessor.named_transformers_['numeric']['impute'].statistics_)

O split estratificado preserva a distribuição de classes, mas não separa motoristas/viagens: o CSV não fornece essas chaves. Duplicatas exatas são removidas antes da divisão; dados sintéticos relacionados podem continuar em ambos os lados. Não aplicar SMOTE antes do split ou fora dos folds. Neste experimento não aplicamos SMOTE adicional.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from src.preprocessing import make_pipeline
pipeline = make_pipeline(RandomForestClassifier(random_state=config.seed))
display(pipeline)